In [ ]:
# Example 1: Construct a DBusAddress and build a method-call Message.
from jeepney import DBusAddress, new_method_call

addr = DBusAddress(
    object_path='/org/freedesktop/DBus',
    bus_name='org.freedesktop.DBus',
    interface='org.freedesktop.DBus',
)
print(f"object_path = {addr.object_path}")
print(f"bus_name    = {addr.bus_name}")
print(f"interface   = {addr.interface}")

msg = new_method_call(addr, 'ListNames')
print(f"msg type    = {type(msg).__name__}")
print(f"member      = {msg.header.fields}")
assert msg.header.fields  # method-call has at least PATH/MEMBER fields


In [ ]:
# Example 2: Serialise a Message to bytes and parse it back via Parser.
from jeepney import DBusAddress, HeaderFields, Parser, new_method_call

addr = DBusAddress(
    object_path='/org/freedesktop/DBus',
    bus_name='org.freedesktop.DBus',
    interface='org.freedesktop.DBus',
)
msg = new_method_call(addr, 'GetId')
msg.header.serial = 1

raw = msg.serialise()
print(f"serialised length = {len(raw)} bytes")

parser = Parser()
parser.add_data(raw)
parsed = parser.get_next_message()
print(f"parsed member     = {parsed.header.fields[HeaderFields.member]}")
assert parsed.header.fields[HeaderFields.member] == 'GetId'
assert parsed.header.serial == 1

In [ ]:
# Example 3: MessageGenerator helper builds typed method calls.
from jeepney import MessageGenerator, new_method_call

class DBusProxy(MessageGenerator):
    interface = 'org.freedesktop.DBus'

    def __init__(self):
        super().__init__(
            object_path='/org/freedesktop/DBus',
            bus_name='org.freedesktop.DBus',
        )

    def ListNames(self):
        return new_method_call(self, 'ListNames')

proxy = DBusProxy()
msg = proxy.ListNames()
print(f"interface = {proxy.interface}")
print(f"path      = {proxy.object_path}")
print(f"msg body  = {msg.body}")
assert msg.body == ()


In [ ]:
# Example 4: MatchRule serialises to a D-Bus match rule string.
from jeepney import MatchRule

rule = MatchRule(
    type='signal',
    sender='org.freedesktop.DBus',
    interface='org.freedesktop.DBus',
    member='NameOwnerChanged',
    path='/org/freedesktop/DBus',
)
as_str = rule.serialise()
print(as_str)
assert "type='signal'" in as_str
assert "member='NameOwnerChanged'" in as_str
assert "interface='org.freedesktop.DBus'" in as_str


In [ ]:
# Example 5: bus_messages.message_bus exposes a ready-made proxy for org.freedesktop.DBus.
from jeepney.bus_messages import message_bus

print(f"interface   = {message_bus.interface}")
print(f"bus_name    = {message_bus.bus_name}")
print(f"object_path = {message_bus.object_path}")

hello_msg = message_bus.Hello()
list_names_msg = message_bus.ListNames()
get_id_msg = message_bus.GetId()

for label, m in [('Hello', hello_msg), ('ListNames', list_names_msg), ('GetId', get_id_msg)]:
    print(f"{label:10s} body = {m.body}")

assert message_bus.interface == 'org.freedesktop.DBus'
assert hello_msg.body == ()
print("\njeepney smoke test PASSED")
